In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
import os
# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_DIR = '/content/drive/MyDrive' # Update DATA_DIR to point to Google Drive
TRAIN_FILE = os.path.join(DATA_DIR, 'quickdraw_train.npz')
TEST_FILE = os.path.join(DATA_DIR, 'quickdraw_test.npz')

BATCH_SIZE = 128
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
# ==========================================
# 2. DATASET CLASS (The NPZ Loader)
# ==========================================

class QuickDrawDataset(Dataset):
    def __init__(self, file_path, mode='train'):
        """
        Args:
            file_path (str): Path to the .npz file
            mode (str): 'train' (loads images & labels) or 'test' (loads images only)
        """
        self.mode = mode

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Could not find file: {file_path}")

        print(f"Loading {mode} data from {file_path}...")
        data = np.load(file_path)

        if mode == 'train':
            # Load x_train and y_train
            self.x = data['x_train']
            self.y = data['y_train']
            self.classes = data['class_names']
            print(f"Loaded {len(self.x)} training samples. Classes: {len(self.classes)}")

        elif mode == 'test':
            # Load test_images (for leaderboard inference)
            self.x = data['test_images']
            self.y = None
            print(f"Loaded {len(self.x)} test images.")

        # Pre-processing:
        # Convert to Float Tensor and Normalize (0-255 -> 0-1)
        self.x = torch.from_numpy(self.x).float() / 255.0

        if self.y is not None:
            self.y = torch.from_numpy(self.y).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.mode == 'train':
            label = self.y[idx]
            return img, label
        else:
            return img

In [4]:
# ==========================================
# 3. PREPARE DATALOADERS
# ==========================================

CLASSES = ['apple', 'baseballbat', 'basketball', 'clock', 'compass', 'cookie', 'donut', 'ladder', 'mountain', 'pizza', 'rabbit', 'soccerball', 'spider', 't-shirt', 'wheel']

# 1. Load the Training Data
full_train_dataset = QuickDrawDataset(TRAIN_FILE, mode='train')
NUM_CLASSES = len(full_train_dataset.classes)

# 2. Create Validation Split (80% Train / 20% Val)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# 3. Create Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

Loading train data from /content/drive/MyDrive/quickdraw_train.npz...
Loaded 60000 training samples. Classes: 15
Train samples: 48000 | Validation samples: 12000


In [5]:
# ==========================================
# 4. YOUR IMPLEMENTATION HERE
# ==========================================
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 256
EPOCHS = 20
use_pin = (DEVICE.type == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=use_pin
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_pin
)

In [6]:
# ==========================================
# Part A Pancake
# ==========================================
class PancakeMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, num_classes)
        self.drop = nn.Dropout(0.30)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.fc1(x))
        x = self.drop(x)
        x = torch.relu(self.fc2(x))
        x = self.drop(x)
        return self.fc3(x)

In [7]:
# ==========================================
# Part B Tower
# ==========================================
class TowerMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=NUM_CLASSES):
        super().__init__()
        h = 256
        self.layers = nn.ModuleList([
            nn.Linear(input_size, h),
            nn.Linear(h, h),
            nn.Linear(h, h),
            nn.Linear(h, h),
            nn.Linear(h, h),
            nn.Linear(h, h),
            nn.Linear(h, h),
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(h) for _ in range(7)])
        self.drop = nn.Dropout(0.20)
        self.out = nn.Linear(h, num_classes)

    def forward(self, x):
        x = x.view(-1, 784)
        for i, fc in enumerate(self.layers):
            x = nn.functional.gelu(self.bns[i](fc(x)))
            x = self.drop(x)
        return self.out(x)


In [8]:

def apply_augmentation(X):
    B = X.size(0)
    imgs = X.clone().view(B, 28, 28)

    flip_mask = torch.rand(B, device=imgs.device) < 0.5
    imgs[flip_mask] = imgs[flip_mask].flip(2)

    for i in range(B):
        if torch.rand(1, device=imgs.device) < 0.4:
            h = torch.randint(3, 10, (1,), device=imgs.device).item()
            w = torch.randint(3, 10, (1,), device=imgs.device).item()
            r = torch.randint(0, 28 - h, (1,), device=imgs.device).item()
            c = torch.randint(0, 28 - w, (1,), device=imgs.device).item()
            imgs[i, r:r+h, c:c+w] = 0.0

    X_aug = imgs.view(B, 784)
    X_aug = (X_aug + 0.02 * torch.randn_like(X_aug)).clamp(0, 1)
    return X_aug


def train_one_epoch(model, loader, criterion, optimizer, use_augment=False):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        if use_augment:
            X = apply_augmentation(X)

        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out = model(X)
        loss = criterion(out, y)

        total_loss += loss.item() * X.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs, lr, weight_decay, model_name, use_augment=False, save_path=None):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    warmup = 2
    def lr_lambda(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        progress = (ep - warmup) / max(1, epochs - warmup)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    start = time.time()

    print(f"\n{'='*55}\n{model_name} | Augmentation={'ON' if use_augment else 'OFF'}\n{'='*55}")
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, use_augment)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)

        print(f"Ep {ep:02d} | TrAcc {tr_acc:.2%} | VaAcc {va_acc:.2%} | TrLoss {tr_loss:.4f} | VaLoss {va_loss:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            if save_path:
                torch.save(model.state_dict(), save_path)

    print(f"Done in {time.time()-start:.1f}s | Best Val Acc: {best_val_acc:.2%}")
    return history, best_val_acc


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [9]:
# Train Pancake
pancake = PancakeMLP()
p_hist, p_best = train_model(
    pancake, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=1e-4,
    model_name="Pancake", use_augment=False, save_path="pancake_best.pth"
)



Pancake | Augmentation=OFF
Ep 01 | TrAcc 54.94% | VaAcc 66.34% | TrLoss 1.5589 | VaLoss 1.2527
Ep 02 | TrAcc 67.89% | VaAcc 70.58% | TrLoss 1.1981 | VaLoss 1.1286
Ep 03 | TrAcc 72.33% | VaAcc 73.00% | TrLoss 1.0766 | VaLoss 1.0663
Ep 04 | TrAcc 75.96% | VaAcc 75.02% | TrLoss 0.9867 | VaLoss 1.0124
Ep 05 | TrAcc 78.34% | VaAcc 75.27% | TrLoss 0.9266 | VaLoss 0.9933
Ep 06 | TrAcc 80.27% | VaAcc 75.93% | TrLoss 0.8697 | VaLoss 0.9867
Ep 07 | TrAcc 82.03% | VaAcc 76.87% | TrLoss 0.8219 | VaLoss 0.9723
Ep 08 | TrAcc 83.84% | VaAcc 77.08% | TrLoss 0.7780 | VaLoss 0.9559
Ep 09 | TrAcc 85.57% | VaAcc 77.14% | TrLoss 0.7359 | VaLoss 0.9629
Ep 10 | TrAcc 87.28% | VaAcc 77.49% | TrLoss 0.6923 | VaLoss 0.9562
Ep 11 | TrAcc 88.77% | VaAcc 77.46% | TrLoss 0.6586 | VaLoss 0.9604
Ep 12 | TrAcc 90.18% | VaAcc 78.04% | TrLoss 0.6259 | VaLoss 0.9607
Ep 13 | TrAcc 91.53% | VaAcc 77.87% | TrLoss 0.5954 | VaLoss 0.9533
Ep 14 | TrAcc 92.68% | VaAcc 77.98% | TrLoss 0.5671 | VaLoss 0.9633
Ep 15 | TrAcc 93.71%

In [10]:
# Train Tower
tower = TowerMLP()
t_hist, t_best = train_model(
    tower, train_loader, val_loader, epochs=20, lr=5e-4, weight_decay=1e-4,
    model_name="Tower", use_augment=False, save_path="tower_best.pth"
)

print("\nModel Comparison:")
print(f"Pancake  | Params: {count_params(pancake):,} | Best Val Acc: {p_best:.2%}")
print(f"Tower    | Params: {count_params(tower):,}   | Best Val Acc: {t_best:.2%}")


Tower | Augmentation=OFF
Ep 01 | TrAcc 43.74% | VaAcc 64.11% | TrLoss 1.8287 | VaLoss 1.2341
Ep 02 | TrAcc 65.02% | VaAcc 69.53% | TrLoss 1.2434 | VaLoss 1.1041
Ep 03 | TrAcc 69.60% | VaAcc 71.91% | TrLoss 1.1232 | VaLoss 1.0533
Ep 04 | TrAcc 71.97% | VaAcc 73.25% | TrLoss 1.0587 | VaLoss 1.0146
Ep 05 | TrAcc 74.04% | VaAcc 75.19% | TrLoss 1.0079 | VaLoss 0.9868
Ep 06 | TrAcc 75.45% | VaAcc 75.58% | TrLoss 0.9725 | VaLoss 0.9715
Ep 07 | TrAcc 76.76% | VaAcc 76.28% | TrLoss 0.9337 | VaLoss 0.9599
Ep 08 | TrAcc 78.18% | VaAcc 76.85% | TrLoss 0.9087 | VaLoss 0.9445
Ep 09 | TrAcc 79.03% | VaAcc 77.00% | TrLoss 0.8797 | VaLoss 0.9372
Ep 10 | TrAcc 80.11% | VaAcc 77.59% | TrLoss 0.8528 | VaLoss 0.9331
Ep 11 | TrAcc 80.94% | VaAcc 77.76% | TrLoss 0.8322 | VaLoss 0.9260
Ep 12 | TrAcc 81.54% | VaAcc 77.78% | TrLoss 0.8145 | VaLoss 0.9247
Ep 13 | TrAcc 82.58% | VaAcc 77.98% | TrLoss 0.7915 | VaLoss 0.9237
Ep 14 | TrAcc 83.04% | VaAcc 78.22% | TrLoss 0.7767 | VaLoss 0.9219
Ep 15 | TrAcc 83.36% |

In [11]:
from google.colab import files

files.download('pancake_best.pth')
files.download('tower_best.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# ==========================================
# PART C CHAMPION MODEL
# ==========================================

class ChampionMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=NUM_CLASSES):
        super(ChampionMLP, self).__init__()
        self.fc1   = nn.Linear(input_size, 384)
        self.fc2   = nn.Linear(384, 256)
        self.fc3   = nn.Linear(256, 128)
        self.fc4   = nn.Linear(128, 64)
        self.out   = nn.Linear(64, num_classes)
        self.bn1   = nn.BatchNorm1d(384)
        self.bn2   = nn.BatchNorm1d(256)
        self.bn3   = nn.BatchNorm1d(128)
        self.bn4   = nn.BatchNorm1d(64)
        self.drop1 = nn.Dropout(0.30)
        self.drop2 = nn.Dropout(0.25)
        self.drop3 = nn.Dropout(0.20)
        self.drop4 = nn.Dropout(0.10)
        self._kaiming_init()

    def _kaiming_init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.drop1(nn.functional.gelu(self.bn1(self.fc1(x))))
        x = self.drop2(nn.functional.gelu(self.bn2(self.fc2(x))))
        x = self.drop3(nn.functional.gelu(self.bn3(self.fc3(x))))
        x = self.drop4(nn.functional.gelu(self.bn4(self.fc4(x))))
        return self.out(x)


# --- train ---
BATCH_SIZE = 128
use_pin = (DEVICE.type == "cuda")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=use_pin)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=use_pin)

champion = ChampionMLP()
print(f"Champion params: {count_params(champion):,}")

c_hist, c_best = train_model(
    champion, train_loader, val_loader,
    epochs=35,
    lr=1e-3,
    weight_decay=2e-4,
    model_name="ChampionMLP",
    use_augment=True,
    save_path="champion_best.pth"
)

model = champion
model.load_state_dict(torch.load("champion_best.pth", map_location=DEVICE))
print(f"\nChampion best val: {c_best:.2%}")
print(f"Params: {count_params(champion):,} | Epochs: 35")

Champion params: 443,791

ChampionMLP | Augmentation=ON
Ep 01 | TrAcc 48.99% | VaAcc 66.02% | TrLoss 1.7013 | VaLoss 1.2187
Ep 02 | TrAcc 63.61% | VaAcc 70.04% | TrLoss 1.2921 | VaLoss 1.1090
Ep 03 | TrAcc 67.13% | VaAcc 71.76% | TrLoss 1.1964 | VaLoss 1.0559
Ep 04 | TrAcc 68.85% | VaAcc 73.42% | TrLoss 1.1472 | VaLoss 1.0245
Ep 05 | TrAcc 70.69% | VaAcc 74.62% | TrLoss 1.1060 | VaLoss 0.9966
Ep 06 | TrAcc 71.80% | VaAcc 75.46% | TrLoss 1.0747 | VaLoss 0.9642
Ep 07 | TrAcc 72.91% | VaAcc 76.48% | TrLoss 1.0489 | VaLoss 0.9494
Ep 08 | TrAcc 73.91% | VaAcc 77.14% | TrLoss 1.0233 | VaLoss 0.9324
Ep 09 | TrAcc 74.65% | VaAcc 77.34% | TrLoss 1.0002 | VaLoss 0.9211
Ep 10 | TrAcc 75.46% | VaAcc 77.95% | TrLoss 0.9786 | VaLoss 0.9056
Ep 11 | TrAcc 76.18% | VaAcc 78.45% | TrLoss 0.9641 | VaLoss 0.8924
Ep 12 | TrAcc 77.04% | VaAcc 78.82% | TrLoss 0.9455 | VaLoss 0.8809
Ep 13 | TrAcc 77.46% | VaAcc 79.07% | TrLoss 0.9292 | VaLoss 0.8782
Ep 14 | TrAcc 77.84% | VaAcc 79.05% | TrLoss 0.9182 | VaLoss

In [14]:
import pandas as pd
from sklearn.metrics import accuracy_score

# ==========================================
# 5. INFERENCE & LEADERBOARD VERIFICATION
# ==========================================
print("\n" + "="*40)
print("   GENERATING SUBMISSION FILE")
print("="*40)
# 1. Load Test Images
test_dataset = QuickDrawDataset(TEST_FILE, mode='test')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

def get_predictions(model, loader):
    model.eval()
    model.to(DEVICE)
    preds = []
    with torch.no_grad():
        for batch in loader:
            X = batch.to(DEVICE)
            outputs = model(X)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
    return preds

# 2. Run Inference
print("Running inference on test set...")
predictions = get_predictions(model, test_loader)

# 3. Save as Comma-Separated Text File
submission_file = "submission.txt"
print(f"Saving predictions to '{submission_file}'...")

# Convert list of ints to comma-separated string (e.g., "0,4,9,2...")
submission_string = ",".join(map(str, predictions))

with open(submission_file, "w") as f:
    f.write(submission_string)
print(f"-> Copy & paste the results of this file to the portal.")


   GENERATING SUBMISSION FILE
Loading test data from /content/drive/MyDrive/quickdraw_test.npz...
Loaded 15000 test images.
Running inference on test set...
Saving predictions to 'submission.txt'...
-> Copy & paste the results of this file to the portal.


In [16]:
def print_model_size(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Statistics:")
    print(f"  Total Parameters: {total_params:,}")
    if total_params > 3000000:
        print("  ⚠️ WARNING: You are over the 3M parameter limit!")
    else:
        print("  ✅ Parameter count is within limits.")

print_model_size(model)


Model Statistics:
  Total Parameters: 443,791
  ✅ Parameter count is within limits.
